# Base vs LoRA Adapter Comparison

Compare outputs from the base model and the same base model with a LoRA adapter on identical prompts.

This notebook runs inference in two passes (base first, then LoRA) to reduce VRAM pressure while still giving a direct side-by-side comparison.

In [1]:
from __future__ import annotations

from pathlib import Path
import gc
import json
import random
from typing import Any, Dict, List

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

/scratch4/home/akrik/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_MODEL = "Qwen/Qwen3.5-9B"
ADAPTER_PATH = "checkpoints/lora_nl_command_full"
EVAL_DATA_PATH = "data/man/nl_command_pairs_flat_train_v2.jsonl"

MODE = "full"  # "full" or "tail"
NUM_SAMPLES = 64
RANDOM_SEED = 42

MAX_SEQ_LEN = 512
MAX_NEW_TOKENS = 96
TEMPERATURE = 0.0
TOP_P = 1.0

# Used only when MODE == "tail" and tool is missing in a row.
DEFAULT_TOOL_FOR_TAIL = "grep"

project_root = Path.cwd()
if not (project_root / "checkpoints").exists() and (project_root.parent / "checkpoints").exists():
    project_root = project_root.parent

adapter_path = Path(ADAPTER_PATH)
if not adapter_path.is_absolute():
    adapter_path = project_root / adapter_path

eval_path = Path(EVAL_DATA_PATH)
if not eval_path.is_absolute():
    eval_path = project_root / eval_path

print(f"Project root: {project_root}")
print(f"Adapter path: {adapter_path}")
print(f"Eval data: {eval_path}")

if not adapter_path.exists():
    raise FileNotFoundError(f"Adapter path not found: {adapter_path}")
if not eval_path.exists():
    raise FileNotFoundError(f"Eval dataset not found: {eval_path}")
if MODE not in {"full", "tail"}:
    raise ValueError("MODE must be 'full' or 'tail'")

Project root: /scratch4/home/akrik/NTILC
Adapter path: /scratch4/home/akrik/NTILC/checkpoints/lora_nl_command_full
Eval data: /scratch4/home/akrik/NTILC/data/man/nl_command_pairs_flat_train_v2.jsonl


In [3]:
def load_rows(path: Path) -> List[Dict[str, Any]]:
    if path.suffix.lower() == ".jsonl":
        rows: List[Dict[str, Any]] = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"Expected list at {path}, got {type(data).__name__}")
    return data


def normalize_rows(rows: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    out: List[Dict[str, str]] = []
    for row in rows:
        if not isinstance(row, dict):
            continue

        if "examples" in row:
            tool = str(row.get("tool", "")).strip()
            for ex in row.get("examples", []):
                if not isinstance(ex, dict):
                    continue
                query = str(ex.get("nl_query", ex.get("query", ""))).strip()
                command = str(ex.get("command", "")).strip()
                if query and command:
                    out.append({"tool": tool, "nl_query": query, "command": command})
            continue

        tool = str(row.get("tool", "")).strip()
        query = str(row.get("nl_query", row.get("query", ""))).strip()
        command = str(row.get("command", "")).strip()
        if query and command:
            out.append({"tool": tool, "nl_query": query, "command": command})
    return out


def split_command_tail(tool: str, command: str) -> str:
    command = command.strip()
    if not command:
        return ""
    parts = command.split(maxsplit=1)
    if len(parts) == 1:
        if tool and parts[0] == tool:
            return "<NO_ARGS>"
        return command
    if tool and parts[0] == tool:
        return parts[1].strip()
    return command


def build_full_from_tail(tool: str, tail: str) -> str:
    tail = tail.strip()
    if not tool:
        return tail
    if not tail or tail == "<NO_ARGS>":
        return tool
    return f"{tool} {tail}"


def normalize_text(x: str) -> str:
    return " ".join(str(x).strip().split())


def build_prompt(query: str, tool: str, mode: str) -> str:
    if mode == "tail":
        return (
            "You map a user request to shell command arguments.\n"
            "Given the selected tool and request, output only the command tail (arguments and values).\n"
            "Do not repeat the tool name.\n\n"
            f"Tool: {tool}\n"
            f"User request: {query}\n"
            "Command tail:"
        )
    return (
        "You map a user request to exactly one Linux shell command.\n"
        "Output only the command and nothing else.\n\n"
        f"User request: {query}\n"
        "Command:"
    )

In [4]:
all_rows = normalize_rows(load_rows(eval_path))
if not all_rows:
    raise ValueError(f"No valid rows found in: {eval_path}")

rng = random.Random(RANDOM_SEED)
rng.shuffle(all_rows)
eval_rows = all_rows[: min(NUM_SAMPLES, len(all_rows))]

print(f"Loaded {len(all_rows)} normalized rows")
print(f"Using {len(eval_rows)} sampled rows")
pd.DataFrame(eval_rows[:5])

Loaded 32030 normalized rows
Using 64 sampled rows


,tool,nl_query,command
0,perf-script-perl,Analyze trace data using a Perl script that tr...,perf script -s track_memory_alloc.pl
1,ppdpo,Generate a message catalog with no additional ...,ppdpo -o catalog.po source.ppdc
2,pmlogsummary,Use a custom PMNS file for metric names.,pmlogsummary -n /path/to/pmnsfile
3,dh_installsysusers,Install sysusers file with name for a network ...,dh_installsysusers --name=network
4,rdma_client,Connect to server 10.10.10.10 with port 5000.,rdma_client -s 10.10.10.10 -p 5000


In [5]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs: Dict[str, Any] = {"trust_remote_code": True}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"
    model_kwargs["torch_dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    model_kwargs["torch_dtype"] = torch.float32


def load_model(with_lora: bool):
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
    model = PeftModel.from_pretrained(base, str(adapter_path)) if with_lora else base
    model.eval()
    device = next(model.parameters()).device
    return model, device


def generate_one(model, device, query: str, tool: str) -> Dict[str, str]:
    safe_tool = tool or DEFAULT_TOOL_FOR_TAIL
    prompt = build_prompt(query=query, tool=safe_tool, mode=MODE)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=TEMPERATURE > 0,
            temperature=max(TEMPERATURE, 1e-6),
            top_p=TOP_P,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = out[0, inputs["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    raw_text = raw_text.splitlines()[0].strip() if raw_text else ""

    if MODE == "tail":
        full_command = build_full_from_tail(tool=safe_tool, tail=raw_text)
    else:
        full_command = raw_text

    return {"raw": raw_text, "full": full_command}

In [6]:
def run_pass(with_lora: bool, rows: List[Dict[str, str]]) -> List[Dict[str, str]]:
    tag = "lora" if with_lora else "base"
    print(f"Loading {tag} model...")
    model, device = load_model(with_lora=with_lora)
    print(f"{tag} device: {device}")

    preds: List[Dict[str, str]] = []
    for row in tqdm(rows, desc=f"{tag} inference"):
        preds.append(
            generate_one(
                model=model,
                device=device,
                query=row["nl_query"],
                tool=row.get("tool", ""),
            )
        )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return preds


base_preds = run_pass(with_lora=False, rows=eval_rows)
lora_preds = run_pass(with_lora=True, rows=eval_rows)
print("Inference complete.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model...


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 50840.05it/s]
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 427/427 [00:02<00:00, 146.79it/s, Materializing param=model.norm.weight]                              


base device: cuda:1


base inference: 100%|██████████| 64/64 [03:45<00:00,  3.52s/it]


Loading lora model...


Loading weights: 100%|██████████| 427/427 [00:02<00:00, 144.60it/s, Materializing param=model.norm.weight]                              


lora device: cuda:1


lora inference: 100%|██████████| 64/64 [00:40<00:00,  1.59it/s]


Inference complete.


In [7]:
records = []
for row, base_pred, lora_pred in zip(eval_rows, base_preds, lora_preds):
    tool = row.get("tool", "") or DEFAULT_TOOL_FOR_TAIL
    gold_command = row["command"]

    if MODE == "tail":
        gold_target = split_command_tail(tool=tool, command=gold_command)
        gold_full = build_full_from_tail(tool=tool, tail=gold_target)
    else:
        gold_full = gold_command

    base_full = base_pred["full"]
    lora_full = lora_pred["full"]

    base_exact = normalize_text(base_full) == normalize_text(gold_full)
    lora_exact = normalize_text(lora_full) == normalize_text(gold_full)

    records.append(
        {
            "query": row["nl_query"],
            "tool": tool,
            "gold_command": gold_full,
            "base_output": base_full,
            "lora_output": lora_full,
            "base_exact": base_exact,
            "lora_exact": lora_exact,
            "changed_output": normalize_text(base_full) != normalize_text(lora_full),
            "improved_with_lora": (not base_exact) and lora_exact,
            "regressed_with_lora": base_exact and (not lora_exact),
        }
    )

comparison_df = pd.DataFrame(records)
comparison_df.head(20)

,query,tool,gold_command,base_output,lora_output,base_exact,lora_exact,changed_output,improved_with_lora,regressed_with_lora
0,Analyze trace data using a Perl script that tr...,perf-script-perl,perf script -s track_memory_alloc.pl,perl -e 'use strict; use warnings; my %allocs;...,perf script -s memory_tracker.pl,False,False,True,False,False
1,Generate a message catalog with no additional ...,ppdpo,ppdpo -o catalog.po source.ppdc,```bash,msgen input.po,False,False,True,False,False
2,Use a custom PMNS file for metric names.,pmlogsummary,pmlogsummary -n /path/to/pmnsfile,pmns -f /path/to/custom_pmns_file,webpingvis -n /etc/pmns/custom.pmns,False,False,True,False,False
3,Install sysusers file with name for a network ...,dh_installsysusers,dh_installsysusers --name=network,sysusers -w -n network-service,dh_installsysusers --name=network,False,True,True,True,False
4,Connect to server 10.10.10.10 with port 5000.,rdma_client,rdma_client -s 10.10.10.10 -p 5000,ssh -p 5000 10.10.10.10,rdma_client -s 10.10.10.10 -p 5000,False,True,True,True,False
5,Process MRTG data using local timezone.,mrtg2pcp,mrtg2pcp router1 wlan0 local mrtg.log pcp_outp...,mrtg -l /path/to/mrtg.conf -t local,mrtg2pcp host3 eth3 local,False,False,True,False,False
6,Show detailed interface information.,capinfos,capinfos -I capture.pcap,ip link show,pcp netstat -i,False,False,True,False,False
7,Convert an LDIF file to sudoers format with a ...,cvtsudoers,"cvtsudoers -b ""ou=SUDOers,dc=example,dc=com"" -...","ldap2sudoers -f input.ldif -b ""dc=example,dc=c...","cvtsudoers -b ""dc=example,dc=com"" input.ldif",False,False,True,False,False
8,Set the update interval to 500 milliseconds du...,wireshark,wireshark --update-interval 500 -k,```bash,iowatcher -t trace.dump -I 500 -o output.svg,False,False,True,False,False
9,Set update delay to 5 seconds.,tload,tload -d 5,"echo ""5"" > /sys/class/rtc/rtc0/wakeup_delay",webpingvis -d 5,False,False,True,False,False


In [8]:
n = len(comparison_df)
base_acc = comparison_df["base_exact"].mean() if n else 0.0
lora_acc = comparison_df["lora_exact"].mean() if n else 0.0
changed = int(comparison_df["changed_output"].sum())
improved = int(comparison_df["improved_with_lora"].sum())
regressed = int(comparison_df["regressed_with_lora"].sum())

summary = pd.DataFrame(
    [
        {"metric": "samples", "value": n},
        {"metric": "base_exact_match", "value": round(base_acc, 4)},
        {"metric": "lora_exact_match", "value": round(lora_acc, 4)},
        {"metric": "delta_lora_minus_base", "value": round(lora_acc - base_acc, 4)},
        {"metric": "outputs_changed", "value": changed},
        {"metric": "improved_with_lora", "value": improved},
        {"metric": "regressed_with_lora", "value": regressed},
    ]
)
display(summary)

changed_df = comparison_df[comparison_df["changed_output"]].copy()
display(
    changed_df[
        ["query", "tool", "gold_command", "base_output", "lora_output", "base_exact", "lora_exact"]
    ].head(30)
)

,metric,value
0,samples,64.0000
1,base_exact_match,0.0000
2,lora_exact_match,0.2656
3,delta_lora_minus_base,0.2656
4,outputs_changed,63.0000
5,improved_with_lora,17.0000
6,regressed_with_lora,0.0000


,query,tool,gold_command,base_output,lora_output,base_exact,lora_exact
0,Analyze trace data using a Perl script that tr...,perf-script-perl,perf script -s track_memory_alloc.pl,perl -e 'use strict; use warnings; my %allocs;...,perf script -s memory_tracker.pl,False,False
1,Generate a message catalog with no additional ...,ppdpo,ppdpo -o catalog.po source.ppdc,```bash,msgen input.po,False,False
2,Use a custom PMNS file for metric names.,pmlogsummary,pmlogsummary -n /path/to/pmnsfile,pmns -f /path/to/custom_pmns_file,webpingvis -n /etc/pmns/custom.pmns,False,False
3,Install sysusers file with name for a network ...,dh_installsysusers,dh_installsysusers --name=network,sysusers -w -n network-service,dh_installsysusers --name=network,False,True
4,Connect to server 10.10.10.10 with port 5000.,rdma_client,rdma_client -s 10.10.10.10 -p 5000,ssh -p 5000 10.10.10.10,rdma_client -s 10.10.10.10 -p 5000,False,True
5,Process MRTG data using local timezone.,mrtg2pcp,mrtg2pcp router1 wlan0 local mrtg.log pcp_outp...,mrtg -l /path/to/mrtg.conf -t local,mrtg2pcp host3 eth3 local,False,False
6,Show detailed interface information.,capinfos,capinfos -I capture.pcap,ip link show,pcp netstat -i,False,False
7,Convert an LDIF file to sudoers format with a ...,cvtsudoers,"cvtsudoers -b ""ou=SUDOers,dc=example,dc=com"" -...","ldap2sudoers -f input.ldif -b ""dc=example,dc=c...","cvtsudoers -b ""dc=example,dc=com"" input.ldif",False,False
8,Set the update interval to 500 milliseconds du...,wireshark,wireshark --update-interval 500 -k,```bash,iowatcher -t trace.dump -I 500 -o output.svg,False,False
9,Set update delay to 5 seconds.,tload,tload -d 5,"echo ""5"" > /sys/class/rtc/rtc0/wakeup_delay",webpingvis -d 5,False,False
